# Agentic Loop — Visual Explanation

This notebook explains the exact loop block line by line, with a focus on three things:
1. What happens **before** a line runs
2. What happens **at** that line
3. What the state looks like **after** that line or iteration

It is designed as a visual study aid for understanding how `message_history`, model requests, and tool outputs interact across loop iterations. [cite:322]


## The exact code block

```python
message_history = [
    {'role': 'system', 'content': instructions},
    {'role': 'user', 'content': question}
]

iteration_number = 1

while True:
    response = openai_client.responses.create(
        model='gpt-4o-mini',
        input=message_history,
        tools=[search_tool],
    )

    print(f'iteraration number {iteration_number}...')
    message_history.extend(response.output)

    has_function_calls = False

    for message in response.output:
        if message.type == 'function_call':
            print(f'executing {message.name}({message.arguments})...')
            tool_call_output = make_call(message)
            message_history.append(tool_call_output)
            has_function_calls = True

        if message.type == 'message':
            text = message.content[0].text
            print('ASSISTANT:', text)

    iteration_number = iteration_number + 1
    print()

    if not has_function_calls:
        break
```


## Bird's-eye view

```text
Question + instructions
        |
        v
Call model
        |
        v
Model may request tool(s)
        |
        v
Python executes tool(s)
        |
        v
Tool results appended to message_history
        |
        v
Call model again
        |
        v
No more tool calls? -> break
```

The loop continues only because `message_history` keeps growing with both model outputs and tool outputs. [cite:322]


## Step 0 — Initial state

Before the loop starts, the conversation contains only two items:
- the system instructions,
- the user question.

At this point, there are no search results, no tool calls, and no assistant answer yet.


In [ ]:
message_history = [
    {'role': 'system', 'content': instructions},
    {'role': 'user', 'content': question}
]

# Before this line:
# - We already have `instructions` and `question` variables.
# - Nothing has been sent to the model yet.
#
# At this line:
# - We create the initial conversation history.
#
# After this line:
# - message_history has exactly 2 items:
#   [0] system
#   [1] user
#
# Possible output:
# - No printed output


In [ ]:
iteration_number = 1

# Before this line:
# - No loop cycle has started yet.
#
# At this line:
# - We initialize a counter so we can label each run.
#
# After this line:
# - iteration_number == 1
#
# Possible output:
# - No printed output


## Step 1 — Enter the loop

The loop starts and will keep running until Python sees that the model requested no more tools.


In [ ]:
while True:
    response = openai_client.responses.create(
        model='gpt-4o-mini',
        input=message_history,
        tools=[search_tool],
    )

# Before this block:
# - message_history contains the full conversation so far.
# - In iteration 1, it contains only system + user.
#
# At this block:
# - We send the current history to the model.
# - We also tell the model that `search_tool` is available.
# - The model does NOT directly run search() here.
# - The model only decides whether to request the tool.
#
# After this block:
# - `response.output` may contain:
#   - one or more function_call objects,
#   - one or more message objects,
#   - or both.
#
# Possible output:
# - No printed output unless you display `response`
# - Example internal result: function_call search({"query": "create dashboard"})


## Step 2 — Print iteration number

This line is only for readability. It helps you see which pass through the loop is happening.


In [ ]:
print(f'iteraration number {iteration_number}...')

# Before this line:
# - `iteration_number` is already set.
#
# At this line:
# - We print the current loop number.
#
# After this line:
# - Nothing changes in memory; only notebook output is produced.
#
# Possible output:
# iteraration number 1...


## Step 3 — Save the model's returned objects

This is one of the most important lines. It does **not** append real search results yet. It appends whatever the model returned, including tool requests.


In [ ]:
message_history.extend(response.output)

# Before this line:
# - message_history contains everything from earlier turns only.
# - Example before iteration 1: [system, user]
#
# At this line:
# - We append the model's output objects into message_history.
# - If the model requested a tool, that request itself is now stored in history.
#
# After this line:
# - message_history is bigger.
# - Example after iteration 1 might be:
#   [0] system
#   [1] user
#   [2] function_call search({"query": "create dashboard"})
#
# Possible output:
# - No printed output


## Step 4 — Assume there are no tool calls yet

This variable is the loop's stop-check flag.


In [ ]:
has_function_calls = False

# Before this line:
# - We do not yet know whether this iteration contains tool requests.
#
# At this line:
# - We start with the assumption that there are none.
#
# After this line:
# - has_function_calls == False
# - If we find at least one function_call later, we switch it to True
#
# Possible output:
# - No printed output


## Step 5 — Inspect every returned object

A single model response may contain multiple items, so Python loops through each item one by one.


In [ ]:
for message in response.output:
    if message.type == 'function_call':
        print(f'executing {message.name}({message.arguments})...')
        tool_call_output = make_call(message)
        message_history.append(tool_call_output)
        has_function_calls = True

    if message.type == 'message':
        text = message.content[0].text
        print('ASSISTANT:', text)

# Before this block:
# - response.output already exists.
# - It may contain 1 item or many items.
#
# At this block:
# - Python checks each returned object.
# - If it is a function_call, Python runs the real tool.
# - If it is a message, Python prints the assistant text.
#
# After this block:
# - Any tool results are appended to message_history.
# - has_function_calls becomes True if at least one function_call was found.
#
# Possible output in iteration 1:
# executing search({"query": "create dashboard"})...
#
# Possible output in iteration 2:
# ASSISTANT: I first want to search for the core phrase...
# executing search({"query": "dashboard panels"})...
# executing search({"query": "dashboard report setup"})...


## Step 5A — What `make_call(message)` really does

This is where the real Python function runs. The model asked for a tool, but Python must execute it.


```text
message.type == function_call
        |
        v
message.name == "search"
message.arguments == {"query": "create dashboard"}
        |
        v
make_call(message)
        |
        v
search(query="create dashboard")
        |
        v
returns 5 chunks
        |
        v
pack into function_call_output dict
        |
        v
append to message_history
```


## Step 5B — What gets appended after tool execution

After `make_call(...)`, Python appends a tool-output object, not a plain text string.


Example conceptual state after iteration 1:

```text
message_history
[0] system: instructions
[1] user:   "How do I create a dahsbord in Evidently?"
[2] function_call:        search({"query": "create dashboard"})
[3] function_call_output: 5 search-result chunks as JSON
```


## Step 6 — Increment the counter

This line prepares the label for the next loop cycle.


In [ ]:
iteration_number = iteration_number + 1
print()

# Before this line:
# - The current iteration has already finished processing its messages.
#
# At this line:
# - We increase the counter by 1.
# - We print a blank line for readability.
#
# After this line:
# - If we were at 1, now we are at 2.
#
# Possible output:
# <blank line>


## Step 7 — Decide whether to stop

This line is the actual exit condition for the loop.


In [ ]:
if not has_function_calls:
    break

# Before this block:
# - We already inspected every item in response.output.
# - We know whether at least one tool was requested.
#
# At this block:
# - If no tool calls were found, stop the loop.
#
# After this block:
# - Loop ends only when the model returns no function_call objects.
# - Otherwise the next iteration begins.
#
# Possible output:
# - No printed output directly
# - But the notebook stops looping here if has_function_calls is False


## Visual timeline of 3 example iterations

This is the chapter's teaching example, not a universal rule.


```text
ITERATION 1
Input history:
  [system, user]
Model returns:
  function_call search("create dashboard")
Python appends:
  function_call_output with 5 chunks

ITERATION 2
Input history:
  [system, user, function_call, function_call_output]
Model returns:
  message with reasoning
  function_call search("dashboard panels")
  function_call search("dashboard report setup")
Python appends:
  2 function_call_output objects

ITERATION 3
Input history:
  all prior items
Model returns:
  final assistant message
No function_call found
Loop breaks
```


## Most important conceptual distinction

Do not merge these two ideas in your head:

- `responses.create(...)` = the model **decides** what to do next
- `make_call(...)` = Python **actually executes** the requested tool

The model requests; Python executes; Python appends results; the model reads those results next time. [cite:322]


## Practice prompt

After reading this, try explaining the following in your own words:
1. Why `message_history.extend(response.output)` does not yet mean the search results were appended
2. Why `make_call(message)` is the point where the real Python tool runs
3. Why the loop stops only when no `function_call` objects are returned
